In [ ]:
# !pip install openpyxl

In [ ]:
import re
import os
import pandas as pd

# === FILE PATHS ===
INPUT_CSV = "FGD1_transcript_turns.csv"   # change if your file has a different name
OUTPUT_CSV_CLEAN = "FGD1_transcript_turns_cleaned.csv"

# Preview settings for console output
CLEAN_SAMPLE_ROWS = 10

assert os.path.exists(INPUT_CSV), f"Input file not found: {INPUT_CSV}"

# === LOAD DATA FROM CSV ===
df = pd.read_csv(INPUT_CSV)
print("Loaded CSV.")
print("Columns found:", df.columns.tolist())
print("Number of rows before cleaning:", len(df))

# Make sure we actually have an 'utterance' column
assert "utterance" in df.columns, "Expected an 'utterance' column in the input CSV."


# ---------- 1. Helper: remove filler words ----------

FILLER_PATTERNS = [
    r"\buh\b",
    r"\bum\b",
    r"\berm\b",
    r"\ber\b",
    r"\burr\b",
    r"\bah\b",
    r"\buhm\b",
    r"\bmmm\b",
    r"\byou know\b",
    r"\bkind of\b",
    r"\bsort of\b",
    r"\bi mean\b",
    r"\bi guess\b",
    r"\bright\?\b",
    r"\bok,\b",
    r"\bok.\b",
    r"\boh,\b",
    r"\bso,\b",
    r"\bok\b",
    r"\bokay\b",
    r"\bwell,\b",
]

FILLER_REGEXES = [re.compile(pat, flags=re.IGNORECASE) for pat in FILLER_PATTERNS]


def remove_filler_words(text: str) -> str:
    if not isinstance(text, str):
        return text
    new_text = text
    for regex in FILLER_REGEXES:
        new_text = regex.sub(" ", new_text)
    return new_text


# ---------- 2. Helper: collapse repeated words ----------

def collapse_repeated_words(text: str) -> str:
    """
    Turns:
        "OK OK OK, I think..." -> "OK, I think..."
        "ya ya" -> "ya"
    Only for directly repeated words.
    """
    if not isinstance(text, str):
        return text

    pattern = re.compile(r"\b(\w+)(\s+\1\b)+", flags=re.IGNORECASE)
    return pattern.sub(r"\1", text)


# ---------- 3. Master cleaning function for utterances ----------

def clean_utterance(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = remove_filler_words(text)
    text = collapse_repeated_words(text)

    # Punctuation tidy: drop leading stray punct (even after quotes), trim isolated commas/periods, collapse repeats, drop trailing lone period
    text = re.sub(r"^[^A-Za-z0-9]+(?=[A-Za-z0-9])", "", text)
    text = re.sub(r"^[\"“”']*[.,…!?]+\s*", "", text)
    text = re.sub(r"\s+[,\.]{1,}\s+", " ", text)
    text = re.sub(r"([,\.\.!?…]){2,}", r"\1", text)
    text = re.sub(r"\s*\.$", "", text)  # drop trailing lone period like "word ."

    text = re.sub(r"\s+", " ", text).strip()

    return text


# ---------- 4. Apply cleaning pipeline (no row removals) ----------

# Keep original utterance for audit if not already present
if "utterance_original" not in df.columns:
    df["utterance_original"] = df["utterance"]

# Clean the utterance text in place
df["utterance"] = df["utterance"].apply(clean_utterance)

# Turn IDs stay sequential by session
if "session_id" in df.columns:
    if "turn_id" in df.columns:
        df = df.sort_values(["session_id", "turn_id"]).reset_index(drop=True)
    else:
        df = df.sort_values(["session_id"]).reset_index(drop=True)
    df["turn_id"] = df.groupby("session_id").cumcount() + 1
else:
    df = df.reset_index(drop=True)
    df["turn_id"] = df.index + 1

print("Number of rows after cleaning (no removals):", len(df))
print(f"\nSample of cleaned output (first {CLEAN_SAMPLE_ROWS} rows):")
print(df.head(CLEAN_SAMPLE_ROWS))

# ---------- 5. Save cleaned data ----------

if os.path.exists(OUTPUT_CSV_CLEAN):
    print(f"\nOutput already exists, not overwriting: {OUTPUT_CSV_CLEAN}")
else:
    df.to_csv(OUTPUT_CSV_CLEAN, index=False, encoding="utf-8-sig")
    print(f"\nCleaned data saved to: {OUTPUT_CSV_CLEAN}")
